# Scamtir Backend Server (Native Webcam Streamer)
This backend natively opens your local webcam and streams the live video with YOLO-World Zero-Shot bounding boxes over an HTTP MJPEG stream (`/video_feed`). It also retains the `/detect` endpoint for static image uploads!

In [ ]:
!pip install fastapi uvicorn python-multipart websockets nest-asyncio pydantic ultralytics opencv-python numpy

In [9]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, Form, File, UploadFile
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import json
import time
import random
import threading
import cv2
import numpy as np
from ultralytics import YOLOWorld

app = FastAPI(title="Scamtir Backend API (Native Webcam)")

# Enable CORS for the frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Thread lock to prevent set_classes and predict from colliding
model_lock = threading.Lock()

print("[BACKEND] Loading YOLO-World model...")
model = YOLOWorld("yolov8s-worldv2.pt")
current_classes = ["person", "cell phone", "keyboard"]
model.set_classes(current_classes)
print(f"[BACKEND] \u2705 Model loaded with initial classes: {current_classes}")

# ===== ROBUST WEBCAM INIT (Windows-friendly) =====
cap = None

def open_webcam():
    global cap
    if cap is not None:
        cap.release()
        cap = None
        time.sleep(0.5)
    
    attempts = [
        (0, cv2.CAP_DSHOW,  "index 0 + DirectShow"),
        (0, cv2.CAP_MSMF,   "index 0 + MSMF"),
        (0, cv2.CAP_ANY,    "index 0 + auto"),
        (1, cv2.CAP_DSHOW,  "index 1 + DirectShow"),
        (1, cv2.CAP_ANY,    "index 1 + auto"),
    ]
    
    for idx, backend, desc in attempts:
        print(f"[BACKEND] \U0001f504 Trying webcam: {desc}...")
        test_cap = cv2.VideoCapture(idx, backend)
        time.sleep(0.3)
        if test_cap.isOpened():
            ret, frame = test_cap.read()
            if ret and frame is not None:
                w = int(test_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(test_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                print(f"[BACKEND] \u2705 Webcam opened: {desc} \u2014 Resolution: {w}x{h}")
                cap = test_cap
                return True
            else:
                print(f"[BACKEND] \u26a0\ufe0f  {desc}: opened but cannot read frames.")
                test_cap.release()
        else:
            print(f"[BACKEND] \u274c {desc}: failed to open.")
            test_cap.release()
    
    print("[BACKEND] \u274c ALL webcam attempts failed!")
    print("[BACKEND] \U0001f4a1 Tips: Close other camera apps, shutdown webcam_yolo_world.ipynb kernel, check Privacy settings.")
    return False

print("[BACKEND] Opening native webcam...")
webcam_ok = open_webcam()

# ===== ENDPOINTS =====

@app.get("/health")
def health_check():
    webcam_alive = cap is not None and cap.isOpened()
    status = {
        "status": "ok",
        "model": "yolov8s-worldv2",
        "webcam": webcam_alive,
        "current_classes": current_classes,
        "timestamp": time.time()
    }
    print(f"[BACKEND] Health check pinged. webcam={webcam_alive}, classes={len(current_classes)}")
    return status

@app.post("/retry_webcam")
def retry_webcam():
    print("[BACKEND] \U0001f504 Frontend requested webcam retry...")
    success = open_webcam()
    return {"status": "ok" if success else "failed", "webcam": success}

@app.post("/update_classes")
async def update_classes(vocabs: str = Form(...)):
    global current_classes
    try:
        vocab_list = json.loads(vocabs)
    except:
        vocab_list = [vocabs]
    
    if not vocab_list or len(vocab_list) == 0:
        current_classes = []
        print("[BACKEND] \U0001f4dd Classes cleared (empty list).")
        return {"status": "success", "classes": current_classes}
    
    # Acquire lock so we don't collide with the predict loop
    with model_lock:
        current_classes = vocab_list
        model.set_classes(current_classes)
        print(f"[BACKEND] \U0001f4dd Frontend updated classes to: {current_classes}")
        print(f"[BACKEND] \u2705 model.set_classes() applied with {len(current_classes)} class(es).")
        
    return {"status": "success", "classes": current_classes}

def generate_frames():
    target_fps = 10
    frame_time = 1.0 / target_fps
    frame_count = 0
    print("[BACKEND] \U0001f3a5 MJPEG stream started.")
    
    while True:
        start_time = time.time()
        
        if cap is None or not cap.isOpened():
            time.sleep(1)
            continue
        
        success, frame = cap.read()
        if not success:
            time.sleep(0.1)
            continue
        
        # Acquire lock before predict so set_classes can't fire mid-inference
        with model_lock:
            if len(current_classes) > 0:
                try:
                    results = model.predict(frame, conf=0.1, verbose=False)
                    annotated_frame = results[0].plot()
                    
                    frame_count += 1
                    if frame_count % 30 == 0:
                        detected_indices = results[0].boxes.cls.cpu().tolist()
                        if detected_indices:
                            detected_names = [results[0].names[int(idx)] for idx in detected_indices]
                            counts = {}
                            for name in detected_names:
                                counts[name] = counts.get(name, 0) + 1
                            items_str = ", ".join([f"{k}({v})" for k, v in counts.items()])
                            print(f"[BACKEND] \U0001f50d Frame #{frame_count}: {items_str}")
                        else:
                            print(f"[BACKEND] Frame #{frame_count}: no detections.")
                except RuntimeError as e:
                    print(f"[BACKEND] \u26a0\ufe0f Predict error (class change race): {e}")
                    annotated_frame = frame
            else:
                annotated_frame = frame
            
        ret, buffer = cv2.imencode('.jpg', annotated_frame)
        if not ret:
            continue
            
        frame_bytes = buffer.tobytes()
        yield (b'--frame\r\n'
               b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')
               
        elapsed = time.time() - start_time
        sleep_time = frame_time - elapsed
        if sleep_time > 0:
            time.sleep(sleep_time)

@app.get("/video_feed")
def video_feed():
    print("[BACKEND] \U0001f4e1 New /video_feed connection.")
    return StreamingResponse(generate_frames(), media_type="multipart/x-mixed-replace; boundary=frame")

@app.post("/detect")
async def detect(vocabs: str = Form(...), threshold: float = Form(0.1), file: UploadFile = File(...)):
    try:
        vocab_list = json.loads(vocabs)
    except:
        vocab_list = [vocabs]
        
    print(f"[BACKEND] \U0001f4f8 /detect: vocabs={vocab_list}, threshold={threshold}, file={file.filename}")
    
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    height, width, _ = image.shape
    
    with model_lock:
        model.set_classes(vocab_list)
        results = model.predict(image, conf=threshold, verbose=False)
        # Restore stream classes immediately
        if len(current_classes) > 0:
            model.set_classes(current_classes)
    
    detections = []
    if len(results) > 0 and len(results[0].boxes) > 0:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
            conf = box.conf[0].item()
            cls_idx = int(box.cls[0].item())
            label = results[0].names[cls_idx]
            
            detections.append({
                "id": time.time() + random.random(),
                "label": label,
                "confidence": conf,
                "x": (x1 / width) * 100,
                "y": (y1 / height) * 100,
                "w": ((x2 - x1) / width) * 100,
                "h": ((y2 - y1) / height) * 100
            })
    
    print(f"[BACKEND] \u2705 /detect returned {len(detections)} detection(s).")
    return {"status": "success", "detections": detections}

print("[BACKEND] All endpoints registered: /health, /video_feed, /detect, /update_classes, /retry_webcam")

if __name__ == "__main__":
    print("[BACKEND] \U0001f680 Starting server on http://0.0.0.0:8000")
    try:
        config = uvicorn.Config(app, host="0.0.0.0", port=8000)
        server = uvicorn.Server(config)
        await server.serve()
    finally:
        if cap is not None:
            cap.release()
        print("[BACKEND] Webcam released.")

[BACKEND] Loading YOLO-World model...
[BACKEND] ✅ Model loaded with initial classes: ['person', 'cell phone', 'keyboard']
[BACKEND] Opening native webcam...
[BACKEND] 🔄 Trying webcam: index 0 + DirectShow...
[BACKEND] ✅ Webcam opened: index 0 + DirectShow — Resolution: 640x480
[BACKEND] All endpoints registered: /health, /video_feed, /detect, /update_classes, /retry_webcam
[BACKEND] 🚀 Starting server on http://0.0.0.0:8000


INFO:     Started server process [20496]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


[BACKEND] Health check pinged. webcam=True, classes=3
INFO:     127.0.0.1:31657 - "GET /health HTTP/1.1" 200 OK
[BACKEND] Health check pinged. webcam=True, classes=3
INFO:     127.0.0.1:31657 - "GET /health HTTP/1.1" 200 OK
[BACKEND] 📝 Frontend updated classes to: ['Glasses']
[BACKEND] ✅ model.set_classes() applied with 1 class(es).
INFO:     127.0.0.1:31657 - "POST /update_classes HTTP/1.1" 200 OK
[BACKEND] Health check pinged. webcam=True, classes=1
INFO:     127.0.0.1:31657 - "GET /health HTTP/1.1" 200 OK
[BACKEND] 📝 Frontend updated classes to: ['Glasses']
[BACKEND] ✅ model.set_classes() applied with 1 class(es).
INFO:     127.0.0.1:31657 - "POST /update_classes HTTP/1.1" 200 OK
[BACKEND] 📡 New /video_feed connection.
[BACKEND] 📝 Frontend updated classes to: ['Glasses']
[BACKEND] ✅ model.set_classes() applied with 1 class(es).
INFO:     127.0.0.1:43983 - "POST /update_classes HTTP/1.1" 200 OK
INFO:     127.0.0.1:56806 - "GET /video_feed?t=1777644023133 HTTP/1.1" 200 OK
[BACKEND] 📡 

INFO:     Shutting down
INFO:     Waiting for connections to close. (CTRL+C to force quit)
INFO:     Finished server process [20496]


[BACKEND] Webcam released.
